# LogVar2FJ 3 — a listed chain, and the fit it supports

Notebooks 1 and 2 ran on a world the model owns. This one runs on a real listed option chain:
Euro Stoxx 50, pulled from Bloomberg on 2026-09-15, emitted as a `LogVar2FJModelPrices` block and
fitted.

The pull is **banked**. This notebook runs live only when `DERIVUS_LIVE_BLOOMBERG` is set in the
environment, and then only through a session that counts every request, paces them 25 seconds
apart, stops at a hard cap and stops on the first refusal. Unset — which is how it is executed —
it loads `data/sx5e_chain_20260915.json`, which carries the emitted block, the chain's census and
every contract the emitter believed. Index option quotes are market data; nothing about a book,
a deal or a counterparty is anywhere in this notebook.

## The route the emitter takes

1. One **calendar** request. The chain field on the underlying answers its listed members, and
   each member's ticker carries its own expiry, so the calendar is read off the listing rather
   than asked for.
2. For each expiry a pillar claims, **one request per side** — calls, then puts. The chain field
   lists calls unless told puts, and no value lists both; a pillar asked once comes back
   one-sided, and a one-sided strike carries no put-call parity.
3. The members inside a **strike band** that scales with each expiry's horizon, on a grid of one
   listed strike per two percent of moneyness, asked in **contract batches**.

So a chain costs one calendar request, two per claimed pillar, and two or three contract batches.
Every contract is then screened — dead, one-sided, crossed, off-grid, outside the band — and the
census of what was refused and why is written into the block's own source line, beside the
counts. That line is the provenance: it travels with the quotes into every fit that reads them.

In [1]:
import copy, datetime, io, json, logging, math, os, sys, time

HERE = os.path.abspath(os.getcwd())
REPO = HERE if os.path.isdir(os.path.join(HERE, 'derivus')) else os.path.dirname(HERE)
DATA = os.path.join(HERE if os.path.isdir(os.path.join(HERE, 'data')) else
                    os.path.join(REPO, 'notebooks'), 'data')
sys.path.insert(0, REPO)

import torch
import derivus as rf
from derivus.config import CustomJsonEncoder

LIVE = bool(os.environ.get('DERIVUS_LIVE_BLOOMBERG'))
BANKED = os.path.join(DATA, 'sx5e_chain_20260915.json')
print('derivus   ', os.path.dirname(rf.__file__))
print('device    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('mode      ', 'LIVE - DERIVUS_LIVE_BLOOMBERG is set' if LIVE else
      'BANKED - DERIVUS_LIVE_BLOOMBERG is unset, no request will be sent')

derivus    C:\Users\Vretiel\PycharmProjects\derivus\.claude\worktrees\agent-aec710af441821806\derivus
device     NVIDIA GeForce RTX 3090
mode       BANKED - DERIVUS_LIVE_BLOOMBERG is unset, no request will be sent


### The careful session

Every request the emitter makes passes through this. It counts, it paces, it records what it
asked for, it stops at the cap, and a refusal is recorded and re-raised before anything else is
sent. A terminal is a shared, metered resource and a notebook is the last thing that should be
allowed to hammer one.

In [2]:
PAUSE_S, CAP = 25, 20
UNDERLYING = 'SX5E Index'

if LIVE:
    from derivus_bloomberg.equity_chain import (EquityForward, EquityLadder, equity_option_block,
                                                fetch_equity_chain)
    from derivus_bloomberg.errors import BloombergRequestError
    from derivus_bloomberg.session import BloombergSession

    class Careful(BloombergSession):
        """Counts, paces, records and stops."""

        def __init__(self, log):
            super().__init__()
            self.log, self.sent = log, 0

        def _guard(self, kind, securities, fields):
            if self.sent >= CAP:
                raise BloombergRequestError(
                    'care cap of {} requests reached; nothing more sent'.format(CAP))
            if self.sent:
                time.sleep(PAUSE_S)
            self.sent += 1
            entry = {'n': self.sent, 'kind': kind, 'securities': len(securities),
                     'fields': list(fields),
                     'at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
            self.log.append(entry)
            print('request {} {}: {} securities x {} fields'.format(
                self.sent, kind, len(securities), len(fields)), flush=True)
            return entry

        def reference_data_report(self, securities, fields, overrides=None):
            entry = self._guard('reference', securities, fields)
            try:
                return super().reference_data_report(securities, fields, overrides)
            except BloombergRequestError as refused:
                entry['refused'] = str(refused)
                raise

        def bulk_reference_data_report(self, securities, fields, overrides=None):
            entry = self._guard('bulk', securities, fields)
            try:
                return super().bulk_reference_data_report(securities, fields, overrides)
            except BloombergRequestError as refused:
                entry['refused'] = str(refused)
                raise

    print('the live session is defined; it will send at most %d requests %d s apart' % (CAP, PAUSE_S))
else:
    print('no session is constructed; the banked pull is the data')

no session is constructed; the banked pull is the data


In [3]:
def jsonable(value):
    if isinstance(value, (datetime.date, datetime.datetime)):
        return value.isoformat()
    if hasattr(value, '__dataclass_fields__'):
        return {k: jsonable(getattr(value, k)) for k in value.__dataclass_fields__}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    return value


if LIVE:
    forward = EquityForward(underlying_factor='EUR_SX5E', volatility_factor='EUR_SX5E.EUR',
                            discount_rate='EUR-ESTR', dividend_reference='EUR_SX5E', rate=0.0200)
    ladder, log = EquityLadder(quotes_per_expiry=5), []
    as_of = datetime.date.today()
    with Careful(log) as session:
        session.reference_data_report(['SPX Index'], ['PX_LAST'])   # the one-field canary
        chain = fetch_equity_chain(session, UNDERLYING, as_of, ladder=ladder, batch=125)
    census = {}
    for verdict in chain.rejected.values():
        census[verdict] = census.get(verdict, 0) + 1
    name, block_body = equity_option_block(chain, forward, ladder)
    PULL = {'underlying': UNDERLYING, 'as_of': as_of.isoformat(),
            'session': {'requests': log, 'pause_seconds': PAUSE_S, 'cap': CAP},
            'chain': {'spot': chain.spot, 'spot_as_of': jsonable(chain.spot_as_of),
                      'name': chain.name, 'listed': len(chain.contracts) + len(chain.rejected),
                      'believed': len(chain.contracts), 'census': census,
                      'expiries': jsonable(chain.expiries),
                      'contracts': jsonable(chain.contracts), 'rejected': chain.rejected},
            'block': {name: jsonable(block_body)}}
else:
    PULL = json.load(open(BANKED))

CHAIN = PULL['chain']
BLOCK_NAME = list(PULL['block'])[0]
BLOCK = copy.deepcopy(PULL['block'][BLOCK_NAME])
AS_OF, SPOT = PULL['as_of'], CHAIN['spot']
print('%s as at %s' % (CHAIN['name'], AS_OF))
print('spot %.2f last printed %s' % (SPOT, CHAIN['spot_as_of']))
print('%d requests sent, %d s apart, cap %d' % (
    len(PULL['session']['requests']), PULL['session']['pause_seconds'], PULL['session']['cap']))
for entry in PULL['session']['requests']:
    print('  %2d %-10s %4d securities x %2d fields  %s' % (
        entry['n'], entry['kind'], entry['securities'], len(entry['fields']), entry['at_utc']))

Euro Stoxx 50 Pr as at 2026-09-15
spot 6198.07 last printed 2026-09-15
15 requests sent, 25 s apart, cap 20
   1 reference     1 securities x  1 fields  2026-09-15T08:18:03.750113+00:00
   2 bulk          1 securities x  4 fields  2026-09-15T08:18:29.220134+00:00
   3 bulk          1 securities x  1 fields  2026-09-15T08:18:54.692941+00:00
   4 bulk          1 securities x  1 fields  2026-09-15T08:19:20.326392+00:00
   5 bulk          1 securities x  1 fields  2026-09-15T08:19:45.777561+00:00
   6 bulk          1 securities x  1 fields  2026-09-15T08:20:11.255346+00:00
   7 bulk          1 securities x  1 fields  2026-09-15T08:20:36.722644+00:00
   8 bulk          1 securities x  1 fields  2026-09-15T08:21:02.168699+00:00
   9 bulk          1 securities x  1 fields  2026-09-15T08:21:27.596578+00:00
  10 bulk          1 securities x  1 fields  2026-09-15T08:21:53.028253+00:00
  11 bulk          1 securities x  1 fields  2026-09-15T08:22:18.502276+00:00
  12 bulk          1 securities x 

### The census

`listed` is what the chain field answered. `believed` is what survived the screens. Everything
between is named, with its reason.

In [4]:
print('%d listed, %d believed' % (CHAIN['listed'], CHAIN['believed']))
for verdict, count in sorted(CHAIN['census'].items(), key=lambda kv: -kv[1]):
    print('  %-22s %5d' % (verdict, count))
print()
print('expiries claimed: %s' % ', '.join(CHAIN['expiries']))

1058 listed, 130 believed
  strike-outside-band      344
  expiry-unclaimed         230
  strike-off-grid          230
  no-open-interest          71
  one-sided                 53

expiries claimed: 2026-12-18, 2027-03-19, 2027-09-17, 2028-09-15


### The block's rungs

Five rungs an expiry, chosen by liquidity and weighted by vega alone: a call and a put at the
money, a call and a put a standard deviation out, and the at-the-money itself. They are quoted as
**premiums** — the terminal's own two-way mids — not as vols, so the forward the fit reads them
against is the fit's own arithmetic and not the vendor's.

In [5]:
rows = BLOCK['instrument']['European_Options']
print('%-12s %9s %6s %12s %10s %10s %9s' % (
    'expiry', 'strike', 'type', 'mid premium', 'bid', 'ask', 'weight'))
for row in sorted(rows, key=lambda r: (r['Expiry_Date']['.Timestamp'], r['Strike'])):
    print('%-12s %9.1f %6s %12.2f %10.1f %10.1f %9.5f' % (
        row['Expiry_Date']['.Timestamp'], row['Strike'], row['Option_Type'],
        row['Quoted_Market_Value'], row['Quoted_Bid'], row['Quoted_Ask'], row['Weight']))

expiry          strike   type  mid premium        bid        ask    weight
2026-12-18      5500.0    Put        53.15       52.7       53.6   0.02305
2026-12-18      5950.0    Put       122.10      121.5      122.7   0.03701
2026-12-18      6200.0    Put       200.95      200.2      201.7   0.04203
2026-12-18      6450.0   Call       102.40      101.7      103.1   0.03854
2026-12-18      7000.0   Call         7.30        7.0        7.6   0.00979
2027-03-19      5300.0    Put        75.10       74.5       75.7   0.03236
2027-03-19      5700.0    Put       129.05      128.3      129.8   0.04537
2027-03-19      6300.0   Call       270.40      269.0      271.8   0.05927
2027-03-19      7000.0   Call        42.25       41.6       42.9   0.03323
2027-03-19      7400.0   Call         9.80        9.4       10.2   0.01347
2027-09-17      5000.0    Put       118.80      117.5      120.1   0.04677
2027-09-17      6000.0    Put       324.65      322.8      326.5   0.07843
2027-09-17      6300.0   

In [6]:
print(BLOCK['instrument']['Quote_Source'])

20 rungs (0sd call/0sd put/1sd call/1sd put/ATM), the best 5 an expiry chosen by liquidity and weighted by vega alone on 20 distinct contracts off the listed SX5E Index chain as at 2026-09-15, 130 contracts believed of 1058 listed (230 expiry-unclaimed, 71 no-open-interest, 53 one-sided, 230 strike-off-grid, 344 strike-outside-band); premiums are the terminal's own two-way mids. Forward: spot 6198.07 (last printed 2026-09-15) carried at r=2.0000% on EUR-ESTR against EUR_SX5E [0.25y declared 0.5249% / chain implies 0.5249%, 0.5y declared -0.0684% / chain implies -0.0684%, 1y declared 1.0234% / chain implies 1.0234%, 2y declared 1.0368% / chain implies 1.0368%]; no leverage pair is declared for SX5E Index, so the fit reads the asset class default; rungs the chain does not list, moved or dropped: 0.25y -> 2026-12-18 (0.2575y), 0.5y -> 2027-03-19 (0.5068y), 1y -> 2027-09-17 (1.005y), 2y -> 2028-09-15 (2.003y), 2y 0sd call EMPTY -> the 1sd call band, 2y 1sd call EMPTY -> the 1sd put band, 3

## The market data the fit needs, built here

A premium is not a vol until something says what the forward is. The block names four factors —
the spot, the discount curve, the dividend yield and a volatility surface — and this notebook
builds them itself rather than borrowing a desk's:

- `EquityPrice.EUR_SX5E` at the spot the pull printed.
- `InterestRate.EUR-ESTR` **flat** at 2.00%, the rate the ladder was placed with.
- `DividendRate.EUR_SX5E` **flat** at the carry the chain itself implies, recomputed below.
- `EquityPriceVol.EUR_SX5E.EUR`, a flat placeholder surface — the fit reads premiums, and the
  surface is read only where a seed or a history would be.

**A desk replaces all four with its own curves.** A flat discount curve and a flat dividend yield
are a stand-in that makes this notebook self-contained; they are not a valuation. What they do
not change is the model, the objective or the report.

### The carry the chain implies

Put-call parity is exact at every strike, so a strike quoted two-sided states the forward:
`C - P = exp(-q.tau).S - exp(-r.tau).K`. Taken as a **median** over the strikes nearest the
forward, because one fat-fingered near-money print that every per-contract screen believes would
otherwise move the carry by percentage points.

In [7]:
RATE = 0.0200
as_of_date = datetime.date.fromisoformat(AS_OF)
two_sided = {}
for contract in CHAIN['contracts']:
    mid = 0.5 * (contract['bid'] + contract['ask'])
    two_sided.setdefault(contract['expiry'], {}).setdefault(contract['strike'], {})[
        contract['option_type']] = mid

implied = {}
print('%-12s %7s %7s %14s' % ('expiry', 'tau', 'pairs', 'parity carry'))
for expiry in sorted(two_sided):
    tau = (datetime.date.fromisoformat(expiry) - as_of_date).days / 365.0
    reads = sorted((abs(k / SPOT - 1.0), -math.log(
        (sides['Call'] - sides['Put'] + math.exp(-RATE * tau) * k) / SPOT) / tau)
        for k, sides in two_sided[expiry].items() if 'Call' in sides and 'Put' in sides)
    near = sorted(q for _, q in reads[:5])
    implied[expiry] = (near[len(near) // 2] if len(near) % 2 else
                       0.5 * (near[len(near) // 2 - 1] + near[len(near) // 2]))
    print('%-12s %7.4f %7d %13.4f%%' % (expiry, tau, len(reads), 100 * implied[expiry]))
CARRY = implied[sorted(implied)[2]]
print('\nflat DividendRate placed at the one-year pillar: %.4f%%' % (100 * CARRY))

expiry           tau   pairs   parity carry
2026-12-18    0.2575      14        0.5249%
2027-03-19    0.5068      18       -0.0684%
2027-09-17    1.0055      21        1.0234%
2028-09-15    2.0027       3        1.0368%

flat DividendRate placed at the one-year pillar: 1.0234%


The four pillars do not agree, and they are not meant to: the December and March expiries sit
either side of a dividend season, so a three-month carry and a six-month carry differ by more
than a percentage point. A flat curve at the one-year pillar is the stand-in this notebook uses,
and the two short expiries wear the difference in their fitted smile. This is exactly the place a
desk's own dividend curve goes.

In [8]:
def curve(rows):
    return {'.Curve': {'meta': [], 'data': rows}}


MARKET = {
    'System Parameters': {'Base_Currency': 'EUR', 'Base_Date': {'.Timestamp': AS_OF}},
    'Model Configuration': {'.ModelParams': {'modeldefaults': {}, 'modelfilters': {}}},
    'Price Factor Interpolation': {'.ModelParams': {'modeldefaults': {}, 'modelfilters': {}}},
    'Price Factors': {
        'FxRate.EUR': {'Domestic_Currency': None, 'Interest_Rate': 'EUR-ESTR', 'Priority': 1,
                       'Spot': 1.0},
        'InterestRate.EUR-ESTR': {'Currency': 'EUR', 'Day_Count': 'ACT_365', 'Sub_Type': None,
                                  'Curve': curve([[0.0027397, RATE], [5.0, RATE]])},
        'EquityPrice.EUR_SX5E': {'Spot': SPOT, 'Currency': 'EUR', 'Interest_Rate': 'EUR-ESTR',
                                 'Issuer': '', 'Respect_Default': 'No', 'Jump_Level': 0.0},
        'DividendRate.EUR_SX5E': {'Currency': 'EUR', 'Floor': None,
                                  'Curve': curve([[0.0027397, CARRY], [5.0, CARRY]])},
        'EquityPriceVol.EUR_SX5E.EUR': {
            'Surface_Type': 'Explicit', 'Moneyness_Rule': 'Sticky_Moneyness',
            'Surface': curve([[0.7, 0.02, 0.18], [0.7, 5.0, 0.18], [1.0, 0.02, 0.18],
                              [1.0, 5.0, 0.18], [1.3, 0.02, 0.18], [1.3, 5.0, 0.18]])}},
    'Price Models': {}, 'Correlations': {}, 'Valuation Configuration': {},
    'Bootstrapper Configuration': {'LogVar2FJModelParameters': {'Prices': 'LogVar2FJModel'}},
    'Market Prices': {BLOCK_NAME: BLOCK}}

for name, value in MARKET['Price Factors'].items():
    print('%-30s %s' % (name, json.dumps(value)[:110]))

FxRate.EUR                     {"Domestic_Currency": null, "Interest_Rate": "EUR-ESTR", "Priority": 1, "Spot": 1.0}
InterestRate.EUR-ESTR          {"Currency": "EUR", "Day_Count": "ACT_365", "Sub_Type": null, "Curve": {".Curve": {"meta": [], "data": [[0.002
EquityPrice.EUR_SX5E           {"Spot": 6198.07, "Currency": "EUR", "Interest_Rate": "EUR-ESTR", "Issuer": "", "Respect_Default": "No", "Jump
DividendRate.EUR_SX5E          {"Currency": "EUR", "Floor": null, "Curve": {".Curve": {"meta": [], "data": [[0.0027397, 0.01023442022001789],
EquityPriceVol.EUR_SX5E.EUR    {"Surface_Type": "Explicit", "Moneyness_Rule": "Sticky_Moneyness", "Surface": {".Curve": {"meta": [], "data": 


## The fit

In [9]:
def fit(market, name):
    """`(the written factor, the report the fit printed)`."""
    job = {'Calc': {
        'Calculation': {'Object': 'BaseValuation', 'Base_Date': {'.Timestamp': AS_OF},
                        'Currency': 'EUR', 'Greeks': 'No', 'MCMC_Simulations': 4096,
                        'Random_Seed': 1},
        'MergeMarketData': {'ExplicitMarketData': market},
        'Deals': {'Reference': 'notebook', 'Tag_Titles': '', 'Deals': {'Children': []}}}}
    buf, root = io.StringIO(), logging.getLogger()
    saved, level = root.handlers[:], root.level
    root.handlers, root.level = [logging.StreamHandler(buf)], logging.INFO
    try:
        cx = rf.Context()
        cx.load_json((json.dumps(job, cls=CustomJsonEncoder), name + '.json'))
        cx.bootstrap()
    finally:
        root.handlers, root.level = saved, level
    return cx.current_cfg.params['Price Factors'].get(
        BLOCK_NAME.replace('ModelPrices', 'ModelParameters')), buf.getvalue()


def show(report, *markers):
    for line in report.splitlines():
        if any(marker in line for marker in markers):
            print(line.strip())


started = time.time()
factor, report = fit(MARKET, 'sx5e')
print('the fit took %.1f s' % (time.time() - started))
show(report, 'price by QUADRATURE', 'stage 2 (', 'stage 3 (', 'stage 4 (', 'stage 6 joint',
     'evaluations and')

the fit took 22.3 s
LogVar2FJModelPrices.EUR_SX5E: the vanilla rows price by QUADRATURE, 24 clock nodes x 16 mixer nodes and no draws
stage 2 (alpha, beta): 17 rows, 4 evaluations, residual 4.6193e-03, 2.5s
stage 3 (rho_s, sigma_s): 17 rows, 8 evaluations, residual 4.3614e-03, 6.3s
stage 4 (rho_l, sigma_l): 17 rows, 6 evaluations, residual 2.0404e-03, 4.7s
stage 6 joint polish: 27 rows, 7 evaluations, residual 5.3101e-03, 7.2s
25 evaluations and 25 Jacobians in 20.9s; the inner bootstrap cost 399 pillar passes, each carrying the backward its Newton slope is - 8.0 a sweep over 4 pillars. The seconds go 1.8 on those passes, 13.3 on the Jacobians and 5.9 on everything else


### The curve, and what it reproduced

In [10]:
show(report, 'market ATM^2')
print()
show(report, 'vol points over', 'vol points unweighted', 'wing 70-80%', 'convexity 110-120%')

0.000-0.258y  market ATM^2 16.62% vol, var-swap strip 18.15% vol, fitted xi 18.06% vol (+1.45 / -0.08 vol points)
0.258-0.507y  market ATM^2 17.96% vol, var-swap strip 18.41% vol, fitted xi 22.28% vol (+4.32 / +3.86 vol points)
0.507-1.005y  market ATM^2 16.94% vol, var-swap strip 19.53% vol, fitted xi 20.76% vol (+3.82 / +1.23 vol points)
1.005-2.003y  market ATM^2 18.93% vol, var-swap strip 20.14% vol, fitted xi 21.96% vol (+3.03 / +1.82 vol points)

0.258y  RMSE 1.010 vol points over 5 quotes, worst -1.571 at 113% of forward
0.507y  RMSE 0.794 vol points over 5 quotes, worst +1.481 at 92% of forward
1.005y  RMSE 0.230 vol points over 5 quotes, worst -0.348 at 107% of forward
2.003y  RMSE 0.396 vol points over 5 quotes, worst +0.677 at 142% of forward
RMSE 0.682 vol points unweighted over 20 quotes, 0.525 vega-weighted (the objective's own); the bootstrap's ATM misses -8.3e-15, -3.6e-15, +0.0e+00, -2.2e-15
wing 70-80% residual: +0.375 vol points RMS, worst -0.432
convexity 110-120% r

### Identification on this ladder

Four expiries out to two years, twenty quotes. The slow pair is fitted here where the world
ladder of notebook 1 could not reach it — the two-year pillar is in the block — and the report
says so by printing a stage 4 and a prior row of zero on `Rho_L` and `Sigma_L`. The residual is
the other way: the shortest expiry is a quarter out, past the horizon inside which a wing reaches
the tail, so `Alpha` is entirely its prior row, and the report names that too.

In [11]:
show(report, 'identification, ', 'multiple of ONE quote row')

identification, 2 (alpha, beta): singular values 1.384e+00  2.926e-01; column norms Alpha[0y] 2.52e-05, Beta[0y] 3.52e-03
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Alpha[0y] 12.6x, Beta[0y] 7.94x
identification, 3 (rho_s, sigma_s): singular values 1.351e+00  4.173e-01; column norms Rho_S[0y] 1.17e-02, Sigma_S[0y] 3.44e-03
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_S[0y] 8.21x, Sigma_S[0y] 5.14x
identification, 4 (rho_l, sigma_l): singular values 1.037e+00  9.619e-01; column norms Rho_L 3.03e-02, Sigma_L 1.34e-02
the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_L 0x, Sigma_L 0x
identification, 6 joint polish: singular values 2.136e+00  1.048e+00  4.944e-01  2.801e-01  1.092e-01  6.481e-02; column norms Sigma_L 1.44e-02, Rho_L 3.51e-02, Rho_S[0y] 1.64e-02, Beta[0y] 4.95e-03, Sigma_S[0y] 4.91e-03, Alpha[0y] 3.27e-05
the PRIOR rows there, each as a multiple of ONE quote row 

In [12]:
show(report, 'leverage prior, TWO rows', 'CLASS PRIOR', 'the residual is not identified',
     'pinned:', 'ON GUARD', 'stationary log-vol sd', 'residual (alpha, beta)')

residual (alpha, beta) (48.205, -20.961); leverage products rho_s*sigma_s -1.819, rho_l*sigma_l -0.602; c 0.362, conditioning share gamma^2/alpha^2 0.811, c_eff 0.293
leverage prior, TWO rows - rho_s -0.7000 from the EquityPrice class default; the product rho_s*sigma_s -1.9000 from the EquityPrice class default - at weights 0.02 on rho_s and 0.00833333 on the product, and one standard error of ANY prior row costs 0.00224, which is what one quote missing by one vol point costs on this ladder. The rho_s row also signs the seed and stage 4
stationary log-vol sd 0.893; corner 7.294 with 0.00e+00 of path-days at or above it
the residual is not identified by this ladder at all, so its rows are the whole of what states it (its shortest wing expiry is 0.257534y against the 0.25y Residual_Horizon) - alpha: CLASS PRIOR +44.0000 at spread 0.5, history uninformative - no history carries an estimate; beta/alpha: CLASS PRIOR -0.5000 at spread 0.2, history uninformative - no history carries an estima

### The factor

In [13]:
for name in sorted(factor):
    value = factor[name]
    array = getattr(value, 'array', None)
    if array is not None:
        print('%-16s %s' % (name, ', '.join('%g -> %.10g' % (t, v) for t, v in array.tolist())))
    else:
        print('%-16s %s' % (name, value))

Alpha            0 -> 48.20544861
Beta             0 -> -20.96133888
C_Min            0.12
Cap_A            7.294182811989057
Kappa_L          0.5
Kappa_S          6.0
On_Guard         
Property_Aliases None
Residual_Law     NIG
Rho_L            -0.3704855339661314
Rho_S            0 -> -0.7079431661
Sigma_L          1.6249127125707163
Sigma_S          0 -> 2.568926675
Skew_Gradient    0.0313926552834,8.7812618431
Steps_Per_Year   252.0
Stickiness_Band  0.5
Xi_Curve         0 -> 0.03262596083, 0.257534 -> 0.04963655504, 0.506849 -> 0.04311637619, 1.00548 -> 0.04821060297


## The implied-vol surfaces

Beside the listed chain, the terminal answers a gridded implied-vol surface per index: a vol per
tenor and moneyness, already interpolated. `data/surfaces_20260915.json` carries three of them,
captured the same morning.

They are a different kind of object from a chain. A chain is contracts, with bids, asks, open
interest and a census of what was refused; a surface is a vendor's fitted answer with no
provenance attached. Both are useful, and they are not interchangeable.

In [14]:
SURFACES = json.load(open(os.path.join(DATA, 'surfaces_20260915.json')))
print('captured %s' % SURFACES['as_of_utc'])
print('asked   : tenors %s months by moneyness %s%%' % (
    SURFACES['asked']['tenor_months'], SURFACES['asked']['moneyness_percent']))
for security, body in SURFACES['indices'].items():
    print('\n%s  spot %s, last update %s' % (security, body['spot'], body['last_update']))
    tenors = list(body['answered'])
    moneyness = sorted({k for grid in body['answered'].values() for k in grid}, key=float)
    print('  %-8s %s' % ('tenor', ''.join('%9s%%' % k for k in moneyness)))
    for tenor in tenors:
        print('  %-8s %s' % (tenor, ''.join(
            '%10.4f' % body['answered'][tenor][k] if k in body['answered'][tenor] else '%10s' % '-'
            for k in moneyness)))
    asked_t = set(SURFACES['asked']['tenor_months'])
    asked_k = set(SURFACES['asked']['moneyness_percent'])
    print('  of %d fields asked, %d answered; never answered: tenors %s months, moneyness %s%%'
          % (len(asked_t) * len(asked_k),
             sum(len(grid) for grid in body['answered'].values()),
             sorted(asked_t - {int(t[:-1]) for t in body['answered']}),
             sorted(asked_k - {float(k) for grid in body['answered'].values() for k in grid})))

captured 2026-09-15T07:36:34.669586+00:00
asked   : tenors [1, 2, 3, 6, 9, 12, 18, 24, 36] months by moneyness [80.0, 90.0, 95.0, 100.0, 105.0, 110.0, 120.0]%

SPX Index  spot 7619.98, last update 2026-09-14
  tenor           90%       95%      100%      105%      110%
  3m          21.6498   18.0846   14.4246   12.1384   11.3651
  6m          20.8171   18.2791   15.4349   13.4532   12.3281
  12m         20.8707   19.1967   16.9628   15.3587   14.0942
  18m         21.0602   19.7399   17.8461   16.3750   15.3374
  24m         21.2994   20.1910   18.4729   17.0198   16.1348
  of 63 fields asked, 25 answered; never answered: tenors [1, 2, 9, 36] months, moneyness [80.0, 120.0]%

NDX Index  spot 29127.16, last update 2026-09-14
  tenor           90%       95%      100%      105%      110%
  3m          25.8201   23.0774   20.6065   18.4934   17.3538
  6m          25.1589   23.3575   21.4523   19.9829   19.0345
  12m         25.2001   24.0122   22.5868   21.4748   20.7667
  18m         25.

### What they can and cannot supply

They can supply a **term structure and a smile from three months out**: 3, 6, 12, 18 and 24
months by 90 to 110 percent of moneyness, which is a usable at-the-money strip and a usable
near-money skew for a ladder whose front pillar is a quarter.

They cannot supply:

- **A one- or two-month tenor.** Those fields are not entitled on this workstation, and they are
  exactly the tenors notebook 2 showed the residual's tail is identified at. Whatever else the
  surfaces are good for, they cannot close the model's flat direction.
- **Wings.** 90 to 110 percent is the near-money smile. The convexity that `Alpha` prices sits
  outside it.
- **Anything past two years.** A three-year pillar comes from the listed chain or from nowhere.
- **Provenance.** There is no bid, no ask, no open interest and no census; a surface point cannot
  be traced to the contracts behind it, so a fit that reads one cannot say which quote it is
  missing by a vol point.

The chain is the primary source and the surface is a cross-check, not the other way round.

## What this notebook establishes

- A listed chain can be turned into a fitted factor in one pass, and every quote in it carries
  the census of the chain it came from in the block's own source line.
- The emitter's request budget is bounded and stated: one calendar request, two per claimed
  pillar, two or three contract batches, all paced and capped.
- The market data a premium-quoted block needs is small enough to build in a notebook, and what
  is built here is a stand-in a desk replaces with its own curves.
- On this ladder the slow pair is fitted and the residual's tail is not, and the report says
  which is which without being asked.